# DINOv3 四批编码结果统一聚类与按景点分布分析

这个 notebook 用于将四个已经完成 DINOv3 编码的输出目录合并，进行统一聚类，并按景点 / 公园统计各视觉聚类的分布。

## 适用输入

默认读取以下四个输出目录：

```text
/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid
/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_xs_all
/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_yhy
/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_ymy
```

每个目录应包含：

```text
manifest.csv
features.h5
```

其中 `features.h5` 中优先读取 `embeddings_l2`，如果没有则读取 `embeddings`。

## 输出

统一输出到：

```text
/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv所有生成数据
```

主要结果包括：

```text
all_image_level_joint_clusters.csv
attraction_kmeans_distribution_ratio_wide.csv
attraction_kmeans_distribution_count_wide.csv
attraction_dominant_visual_cluster.csv
cluster_top_attractions.csv
figures/
cluster_sample_grids/
```

In [ ]:
!fusermount -uz /content/drive
!rm -rf /content/drive/*
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import h5py
from pathlib import Path
from google.colab import drive

# 1. Ensure Google Drive is properly mounted by forcing a remount
try:
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    # If the mountpoint is corrupted, unmount and try again
    os.system('fusermount -uz /content/drive')
    drive.mount('/content/drive', force_remount=True)

# 2. Specify the correct path to the H5 file
h5_path = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid/features.h5")

# 3. Check if file exists to prevent FileNotFoundError
if not h5_path.exists():
    print(f"File not found: {h5_path}\nPlease ensure the file exists in your Google Drive.")
else:
    with h5py.File(h5_path, "r") as hf:
        print("H5 keys:", list(hf.keys()))

        print("\nRoot attributes:")
        for k, v in hf.attrs.items():
            print(f"  {k}: {v}")

        print("\nDatasets:")
        def inspect(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"\nDataset: {name}")
                print("  shape:", obj.shape)
                print("  dtype:", obj.dtype)

                if obj.attrs:
                    print("  attributes:")
                    for k, v in obj.attrs.items():
                        print(f"    {k}: {v}")

        hf.visititems(inspect)


In [ ]:
import h5py
import numpy as np

with h5py.File(h5_path, "r") as hf:
    x_raw = hf["embeddings"][0]
    x_l2 = hf["embeddings_l2"][0]

    print("Raw norm:", np.linalg.norm(x_raw))
    print("L2 norm:", np.linalg.norm(x_l2))
    print("Same direction:",
          np.allclose(
              x_raw / np.linalg.norm(x_raw),
              x_l2,
              atol=1e-5
          ))

## 1. 安装依赖

In [ ]:
import os, signal

# 固定一组在 Colab Python 3.12 下相互兼容的二进制包版本
# 运行本格后，下一格会自动重启 runtime；重启后从“挂载 Google Drive”继续运行即可。
!pip install -q --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.2 \
  h5py==3.11.0 \
  pillow==10.4.0 \
  tqdm==4.66.5 \
  matplotlib==3.9.2 \
  umap-learn==0.5.6 \
  hdbscan==0.8.38.post2

# 必须重启一次，否则旧的 numpy/pandas 二进制模块仍然留在内存里。
# Colab 会显示 runtime disconnected，这是正常的。
os.kill(os.getpid(), signal.SIGKILL)

In [ ]:
# 必须重启一次，否则旧的 numpy/pandas 二进制模块仍然留在内存里。
# Colab 会显示 runtime disconnected，这是正常的。
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


## 2. 挂载 Google Drive 并设置路径

如果刚刚执行了安装依赖和重启 runtime，从这里继续运行。注意：香山使用最终补全后的 `DINOv3_park_analysis_uid_xs_all`。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os, json, h5py, gc, math, shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import umap
import hdbscan

# =========================
# 四批 DINOv3 编码结果目录
# =========================
SOURCE_DIRS = {
    "all_parks": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid"),
    "xiangshan_all": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_xs_all"),
    "yuanmingyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_ymy"),
    "yiheyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_yhy"),
}

# 统一聚类输出目录
OUT_DIR = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv所有生成数据")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

SAMPLE_DIR = OUT_DIR / "cluster_sample_grids"
SAMPLE_DIR.mkdir(exist_ok=True)

# 景点 / 公园字段
# 通常原 notebook 中 park_name 对应第一层目录。
# 如果你的真实景点名在其他字段里，例如 parent_folder，可改为：
# ATTRACTION_COL = "parent_folder"
ATTRACTION_COL = "park_name"

RANDOM_STATE = 42

print("输出目录:", OUT_DIR)

## 3. 检查四个输出目录是否完整

In [ ]:
def check_source_dir(tag, out_dir: Path):
    manifest_csv = out_dir / "manifest.csv"
    features_h5 = out_dir / "features.h5"

    if not manifest_csv.exists():
        raise FileNotFoundError(f"[{tag}] 缺少 manifest.csv: {manifest_csv}")
    if not features_h5.exists():
        raise FileNotFoundError(f"[{tag}] 缺少 features.h5: {features_h5}")

    print(f"[OK] {tag}")
    print(" manifest:", manifest_csv)
    print(" h5      :", features_h5)
    print(" h5 size :", round(features_h5.stat().st_size / 1024**3, 3), "GB")


for tag, d in SOURCE_DIRS.items():
    check_source_dir(tag, d)

## 4. 读取并合并四批 manifest 与 embedding

这里会自动：
- 优先读取 `embeddings_l2`
- 检查 `done`
- 删除非有限值和全零向量
- 给每张图生成 `global_uid`
- 合并四批结果

In [ ]:
import h5py # Ensure h5py is imported in this cell

def load_one_source(tag, out_dir: Path):
    manifest_csv = out_dir / "manifest.csv"
    features_h5 = out_dir / "features.h5"

    manifest = pd.read_csv(manifest_csv, low_memory=False)

    with h5py.File(features_h5, "r") as hf:
        print(f"\n[{tag}] H5 keys:", list(hf.keys()))

        if "embeddings_l2" in hf:
            emb_key = "embeddings_l2"
        elif "embeddings" in hf:
            emb_key = "embeddings"
        else:
            raise KeyError(f"[{tag}] H5 中找不到 embeddings_l2 或 embeddings")

        X_all = hf[emb_key][:]
        if "done" in hf:
            done = hf["done"][:].astype(bool)
        else:
            done = np.ones(X_all.shape[0], dtype=bool)

    n = min(len(manifest), X_all.shape[0])
    manifest = manifest.iloc[:n].copy()
    X_all = X_all[:n]
    done = done[:n]

    finite_mask = np.isfinite(X_all).all(axis=1)
    nonzero_mask = np.linalg.norm(X_all, axis=1) > 0
    valid_mask = done & finite_mask & nonzero_mask

    df = manifest.loc[valid_mask].copy().reset_index(drop=True)
    X = X_all[valid_mask].astype("float32")

    df["source_tag"] = tag
    df["source_out_dir"] = str(out_dir)

    # 避免四批 uid 撞车
    if "uid" in df.columns:
        df["global_uid"] = df["source_tag"].astype(str) + "::" + df["uid"].astype(str)
    else:
        df["global_uid"] = df["source_tag"].astype(str) + "::" + df.index.astype(str)

    print(f"[{tag}] manifest rows = {len(manifest):,}")
    print(f"[{tag}] valid embeddings = {len(df):,}")
    print(f"[{tag}] feature shape = {X.shape}")

    return df, X


dfs = []
Xs = []

for tag, out_dir in SOURCE_DIRS.items():
    df_i, X_i = load_one_source(tag, out_dir)
    dfs.append(df_i)
    Xs.append(X_i)

all_df = pd.concat(dfs, ignore_index=True)
X = np.vstack(Xs).astype("float32")

print("\n==============================")
print("合并后图片数:", len(all_df))
print("合并后特征矩阵:", X.shape)
print("可用字段:", list(all_df.columns))
if ATTRACTION_COL in all_df.columns:
    print("景点数量:", all_df[ATTRACTION_COL].nunique())
else:
    print(f"警告：找不到 ATTRACTION_COL = {ATTRACTION_COL}")
print("==============================")

## 5. 去除完全重复路径并保存合并清单

如果四批中有完全相同的 `abs_path`，这里会去重。

In [ ]:
# 如果四批里面有完全相同 abs_path，去重
if "abs_path" in all_df.columns:
    before = len(all_df)
    keep_idx = ~all_df["abs_path"].duplicated()
    all_df = all_df.loc[keep_idx].reset_index(drop=True)
    X = X[keep_idx.values]
    print(f"按 abs_path 去重: {before:,} -> {len(all_df):,}")
else:
    print("manifest 中没有 abs_path 字段，跳过去重。")

# 检查景点字段
if ATTRACTION_COL not in all_df.columns:
    raise KeyError(
        f"找不到景点字段 {ATTRACTION_COL}。\n"
        f"现有字段包括：{list(all_df.columns)}\n"
        "请把 ATTRACTION_COL 改成正确字段，例如 parent_folder。"
    )

# 保存合并后的 manifest
merged_manifest_csv = OUT_DIR / "all_sources_merged_manifest_valid.csv"
all_df.to_csv(merged_manifest_csv, index=False, encoding="utf-8-sig")

print("最终参与聚类图片数:", len(all_df))
print("最终 X shape:", X.shape)
print("已保存:", merged_manifest_csv)

display(all_df[[ATTRACTION_COL, "source_tag", "rel_path", "abs_path"]].head())

### 景点名称清理与统一

根据反馈，部分景点名称可能存在格式不统一的情况（例如大小写、空格与下划线）。运行以下代码统一景点名称，确保相似的景点名称被统计在一起。主要操作包括：
- 将景点名称转换为小写。
- 将下划线替换为空格。
- 去除多余的空格。

In [ ]:
print(f"清理前，景点数量: {all_df[ATTRACTION_COL].nunique()}")

# 转换成字符串，处理非字符串值
all_df[ATTRACTION_COL] = all_df[ATTRACTION_COL].astype(str)

# 转换为小写，替换下划线为空格，并去除多余空格
all_df[ATTRACTION_COL] = (
    all_df[ATTRACTION_COL]
    .str.lower()
    .str.replace('_', ' ')
    .str.replace(r'\s+', ' ', regex=True) # 替换多个空格为单个空格，使用原始字符串避免警告
    .str.strip() # 去除首尾空格
)

print(f"清理后，景点数量: {all_df[ATTRACTION_COL].nunique()}")

# 显示部分清理结果，验证效果
display(all_df[[ATTRACTION_COL, 'source_tag', 'rel_path']].head())

### 景点层级统一化

根据用户需求，将颐和园、圆明园和香山的子景点统一到各自的主公园名称下，并与“all_parks”中的小公园名称保持同一层级。这有助于在更宏观的层面进行分析。

### 过滤掉 '百旺绿地公园'

根据用户需求，在进行 PCA 降维之前，从 `all_df` 和特征矩阵 `X` 中移除 '百旺绿地公园' 的数据。

In [ ]:
print(f"景点层级统一化前，景点数量: {all_df[ATTRACTION_COL].nunique()}")

# 定义主公园名称与 source_tag 的映射
major_park_aggregation_map = {
    "xiangshan_all": "香山-植物园", # 用户指定为“香山-植物园”
    "yuanmingyuan": "圆明园",
    "yiheyuan": "颐和园",
}

# 应用映射来统一景点名称
for source_tag, major_park_name in major_park_aggregation_map.items():
    # 只有当图片来自对应的 source_tag 时才更新 ATTRACTION_COL
    all_df.loc[all_df["source_tag"] == source_tag, ATTRACTION_COL] = major_park_name

print(f"景点层级统一化后，景点数量: {all_df[ATTRACTION_COL].nunique()}")

# 显示部分结果，验证效果
display(all_df[[ATTRACTION_COL, 'source_tag', 'rel_path']].head())

In [ ]:
try:
    print(f"过滤 '百旺绿地公园' 前，图片数量: {len(all_df):,}")

    # 创建一个 mask，排除 '百旺绿地公园'
    exclude_park_name = "百旺绿地公园"
    keep_mask = all_df[ATTRACTION_COL] != exclude_park_name

    # 应用 mask 到 all_df 和 X
    all_df = all_df[keep_mask].reset_index(drop=True)
    X = X[keep_mask.values]

    print(f"过滤 '百旺绿地公园' 后，图片数量: {len(all_df):,}")
    print(f"过滤 '百旺绿地公园' 后，特征矩阵 X 的形状: {X.shape}")

    # 验证 '百旺绿地公园' 是否已被移除
    if exclude_park_name in all_df[ATTRACTION_COL].unique():
        print(f"警告: '{exclude_park_name}' 仍存在于数据中，请检查！")
    else:
        print(f"'{exclude_park_name}' 已成功从数据中移除。")

    # 显示部分结果，验证效果
    display(all_df[[ATTRACTION_COL, 'source_tag', 'rel_path']].head())

except NameError as e:
    if "name 'X' is not defined" in str(e) or "name 'all_df' is not defined" in str(e):
        print("--------------------------------------------------------------------------------")
        print("错误提示：变量 'X' 或 'all_df' 未定义。")
        print("出现此错误通常是因为运行时已重启，或上游数据加载和预处理步骤未执行。")
        print("请务必按顺序运行以下单元格，以重新初始化数据：")
        print("从 `qyQ_D6fFTdUu` (读取并合并四批 manifest 与 embedding) 开始，")
        print("一直运行到 `904aac44` (景点层级统一化)。")
        print("运行完成后，再重新执行此单元格。")
        print("--------------------------------------------------------------------------------")
    else:
        raise e # Re-raise other NameErrors

### 保存过滤 '百旺绿地公园' 后的结果到 CSV

根据用户需求，将过滤 '百旺绿地公园' 后的 `all_df` DataFrame 保存为 CSV 文件，方便后续查看和分析。

In [ ]:
output_csv_path = OUT_DIR / "all_df_filtered_no_baiwanglvdipark.csv"
all_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print(f"过滤 '百旺绿地公园' 后的数据已保存至 CSV: {output_csv_path}")

### 统计每个景点的各个聚类占比

根据用户需求，对过滤后的 `all_df` 进行统计，计算每个景点中每个聚类所占的比例和数量。

In [ ]:
print('--- 重新执行景点聚类分布计算 ---')

# 重新定义 make_distribution_tables 函数，确保其在当前会话中可用
def make_distribution_tables(df, group_col, cluster_col, out_prefix, output_dir):
    count_df = (
        df.groupby([group_col, cluster_col])
        .size()
        .rename("count")
        .reset_index()
    )

    total_df = (
        df.groupby(group_col)
        .size()
        .rename("total_images")
        .reset_index()
    )

    count_df = count_df.merge(total_df, on=group_col, how="left")
    count_df["ratio"] = count_df["count"] / count_df["total_images"]

    count_wide = count_df.pivot(
        index=group_col,
        columns=cluster_col,
        values="count"
    ).fillna(0).astype(int)

    ratio_wide = count_df.pivot(
        index=group_col,
        columns=cluster_col,
        values="ratio"
    ).fillna(0)

    summary = ratio_wide.copy()
    summary["dominant_cluster"] = ratio_wide.idxmax(axis=1)
    summary["dominant_ratio"] = ratio_wide.max(axis=1)
    summary["total_images"] = total_df.set_index(group_col)["total_images"]

    # Save to XLSX instead of CSV
    count_df.to_excel(output_dir / f"{out_prefix}_long.xlsx", index=False)
    count_wide.to_excel(output_dir / f"{out_prefix}_count_wide.xlsx")
    ratio_wide.to_excel(output_dir / f"{out_prefix}_ratio_wide.xlsx")
    summary.to_excel(output_dir / f"{out_prefix}_summary.xlsx")

    return count_df, count_wide, ratio_wide, summary


# 使用当前会话中的 all_df（已包含 kmeans_k10）和 K_FINAL=10
# 变量 OUT_DIR, ATTRACTION_COL 和 K_FINAL 应该已经在之前的单元格中定义
label_col = f"kmeans_k{K_FINAL}"

if label_col not in all_df.columns:
    print(f"错误: 列 '{label_col}' 在 all_df 中不存在。请确保已执行 K-Means 聚类并已将结果添加到 all_df 中。")
    print("当前 all_df 的列有:", all_df.columns.tolist())
else:
    km_long, km_count_wide, km_ratio_wide, km_summary = make_distribution_tables(
        all_df,
        ATTRACTION_COL,
        label_col,
        f"attraction_kmeans_k{K_FINAL}_distribution_filtered",
        OUT_DIR # Assuming OUT_DIR is already defined
    )

    print(f"\n每个景点的 K={K_FINAL} 聚类占比已计算并保存到 {OUT_DIR} 目录。")
    print("显示景点聚类分布总结 (km_summary) 的前几行：")
    display(km_summary.head())
    print("显示景点聚类比例宽格式 (km_ratio_wide) 的前几行：")
    display(km_ratio_wide.head())

## 6. PCA 降维作为聚类输入

建议主聚类使用 PCA 后的 50 维特征，而不是直接使用 768 维原始 embedding。这样速度更快，也更稳定。

In [ ]:
pca_dim = min(50, X.shape[0] - 1, X.shape[1])

pca = PCA(n_components=pca_dim, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

print("PCA shape:", X_pca.shape)
print("PCA explained variance ratio sum:", round(float(pca.explained_variance_ratio_.sum()), 4))

np.save(OUT_DIR / "X_pca_50.npy", X_pca)

## 7. 自动选择 KMeans 聚类数

默认测试 `k=4` 到 `k=12`，使用 silhouette score 作为参考。

注意：silhouette 只是参考，最终论文中优先选择**可解释性更好**的聚类数。

In [ ]:
K_RANGE = range(4, 13)

# silhouette 太慢，所以最多抽样 10000 张计算
rng = np.random.default_rng(RANDOM_STATE)
sample_n = min(10000, X_pca.shape[0])
sample_idx = rng.choice(X_pca.shape[0], size=sample_n, replace=False)

scores = []

for k in K_RANGE:
    print(f"Testing k={k}...")
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init="auto",
        max_iter=300
    )
    labels = km.fit_predict(X_pca)

    score = silhouette_score(
        X_pca[sample_idx],
        labels[sample_idx],
        metric="euclidean"
    )

    scores.append({"k": k, "silhouette": score})
    print(f"k={k}, silhouette={score:.4f}")

score_df = pd.DataFrame(scores)
score_csv = OUT_DIR / "kmeans_k_selection_scores.csv"
score_df.to_csv(score_csv, index=False, encoding="utf-8-sig")

best_k = int(score_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("Best k =", best_k)
print("已保存:", score_csv)

display(score_df)

In [ ]:
K_RANGE_EXTRA = range(13, 21)

extra_scores = []

for k in K_RANGE_EXTRA:
    print(f"Testing k={k}...")
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init="auto",
        max_iter=300
    )
    labels = km.fit_predict(X_pca)

    score = silhouette_score(
        X_pca[sample_idx],
        labels[sample_idx],
        metric="euclidean"
    )

    extra_scores.append({"k": k, "silhouette": score})
    print(f"k={k}, silhouette={score:.4f}")

extra_score_df = pd.DataFrame(extra_scores)

score_df_all = pd.concat([score_df, extra_score_df], ignore_index=True)
score_df_all = score_df_all.sort_values("k")

score_df_all.to_csv(
    OUT_DIR / "kmeans_k_selection_scores_k4_to_k20.csv",
    index=False,
    encoding="utf-8-sig"
)

display(score_df_all)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(score_df_all["k"], score_df_all["silhouette"], marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Silhouette score")
plt.title("KMeans K selection based on PCA-reduced DINOv3 embeddings")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_k_selection_curve_k4_to_k20.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import umap
from pathlib import Path
import os, math, h5py, gc
import numpy as np


# =========================
# 0. 基础配置
# =========================

SOURCE_DIRS = {
    "all_parks": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid"),
    "xiangshan_all": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_xs_all"),
    "yuanmingyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_ymy"),
    "yiheyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_yhy"),
}

# 新增：图片文件实际的根目录，`manifest.csv` 中的 `rel_path` 字段是相对于这个目录的相对路径。
IMAGE_ROOT_DIR = Path("/content/drive/MyDrive/CES分析/图片")

OUT_DIR = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv生成数据2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

SAMPLE_DIR = OUT_DIR / "cluster_sample_grids"
SAMPLE_DIR.mkdir(exist_ok=True)

ATTRACTION_COL = "park_name"
RANDOM_STATE = 42

# 修正：将 K_CANDIDATES 设置为只包含用户选择的 K=6
K_CANDIDATES = [6]

print("输出目录:", OUT_DIR)
print("图片文件根目录:", IMAGE_ROOT_DIR)

# =========================
# 1. 检查四批结果是否存在
# =========================

for tag, out_dir in SOURCE_DIRS.items():
    manifest_csv = out_dir / "manifest.csv"
    features_h5 = out_dir / "features.h5"

    print("\n检查:", tag)
    print("manifest:", manifest_csv, "exists =", manifest_csv.exists())
    print("features :", features_h5, "exists =", features_h5.exists())

    if not manifest_csv.exists():
        raise FileNotFoundError(f"[{tag}] 缺少 manifest.csv: {manifest_csv}")
    if not features_h5.exists():
        raise FileNotFoundError(f"[{tag}] 缺少 features.h5: {features_h5}")

# =========================
# 2. 读取单批 manifest + embedding
# =========================

def load_one_source(tag, out_dir: Path):
    manifest_csv = out_dir / "manifest.csv"
    features_h5 = out_dir / "features.h5"

    manifest = pd.read_csv(manifest_csv, low_memory=False)

    with h5py.File(features_h5, "r") as hf:
        print(f"\n[{tag}] H5 keys:", list(hf.keys()))

        if "embeddings_l2" in hf:
            emb_key = "embeddings_l2"
        elif "embeddings" in hf:
            emb_key = "embeddings"
        else:
            raise KeyError(f"[{tag}] H5 中找不到 embeddings_l2 或 embeddings")

        X_all = hf[emb_key][:]

        if "done" in hf:
            done = hf["done"][:].astype(bool)
        else:
            done = np.ones(X_all.shape[0], dtype=bool)

    n = min(len(manifest), X_all.shape[0])
    manifest = manifest.iloc[:n].copy()
    X_all = X_all[:n]
    done = done[:n]

    finite_mask = np.isfinite(X_all).all(axis=1)
    nonzero_mask = np.linalg.norm(X_all, axis=1) > 0
    valid_mask = done & finite_mask & nonzero_mask

    df = manifest.loc[valid_mask].copy().reset_index(drop=True)
    X = X_all[valid_mask].astype("float32")

    df["source_tag"] = tag
    df["source_out_dir"] = str(out_dir)

    if "uid" in df.columns:
        df["global_uid"] = df["source_tag"].astype(str) + "::" + df["uid"].astype(str)
    else:
        df["global_uid"] = df["source_tag"].astype(str) + "::" + df.index.astype(str)

    print(f"[{tag}] manifest rows     = {len(manifest):,}")
    print(f"[{tag}] valid embeddings  = {len(df):,}")
    print(f"[{tag}] feature shape     = {X.shape}")

    return df, X

# =========================
# 3. 合并四批结果
# =========================

dfs = []
Xs = []

for tag, out_dir in SOURCE_DIRS.items():
    df_i, X_i = load_one_source(tag, out_dir)
    dfs.append(df_i)
    Xs.append(X_i)

all_df = pd.concat(dfs, ignore_index=True)
X = np.vstack(Xs).astype("float32")

print("\n合并前图片数:", len(all_df))
print("合并前 X shape:", X.shape)

# 按 abs_path 去重
if "abs_path" in all_df.columns:
    before = len(all_df)
    keep_mask = ~all_df["abs_path"].duplicated()
    all_df = all_df.loc[keep_mask].reset_index(drop=True)
    X = X[keep_mask.values]
    print(f"按 abs_path 去重: {before:,} -> {len(all_df):,}")

# --- BEGIN MODIFICATION ---
# 修正 abs_path，使其指向 Google Drive 的正确图片根目录
if "rel_path" in all_df.columns:
    print(f"修正 'abs_path' 列以匹配 Google Drive 图片根目录: {IMAGE_ROOT_DIR}...")

    def build_correct_image_path(row):
        if pd.isna(row["rel_path"]):
            return None

        rel_path_obj = Path(row["rel_path"])
        # The first component of rel_path is typically the park name (e.g., '颐和园', '一亩园公园')
        first_component_of_rel_path = rel_path_obj.parts[0]

        # Determine the intermediate folder name based on the first component
        intermediate_dir_map = {
            "颐和园": "图片下载-颐和园",
            "圆明园": "图片下载-圆明园",
            # Assuming '香山' or '香山-植物园' would be the first part for this category
            "香山": "图片下载-香山-植物园",
            "香山-植物园": "图片下载-香山-植物园",
            "植物园": "图片下载-香山-植物园" # If rel_path starts with just '植物园'
        }

        intermediate_dir = intermediate_dir_map.get(first_component_of_rel_path, "图片下载-公园")

        # Construct the full absolute path
        return str(IMAGE_ROOT_DIR / intermediate_dir / rel_path_obj)

    all_df["abs_path"] = all_df.apply(build_correct_image_path, axis=1)
    print("'abs_path' 列修正完成。")
elif "source_out_dir" in all_df.columns:
    # Fallback if rel_path is missing, but source_out_dir exists
    # This part was part of previous solution and can be kept as fallback.
    print(f"警告: 'rel_path' 列缺失，尝试从 'source_out_dir' 和 'file_name' 构建 'abs_path'...")
    all_df["abs_path"] = all_df.apply(
        lambda row: str(Path(row["source_out_dir"]).parent / "images" / row["file_name"])
        if pd.notnull(row["source_out_dir"]) and pd.notnull(row["file_name"])
        else None, axis=1
    )
    print("'abs_path' 列通过 'source_out_dir' 和 'file_name' 构建完成。")
else:
    print("警告: 无法修正 'abs_path' 列，因为 'rel_path' 和 'source_out_dir' 均缺失。")
# --- END MODIFICATION ---

# 检查景点字段
if ATTRACTION_COL not in all_df.columns:
    raise RuntimeError(
        f"all_df 里找不到景点字段 ATTRACTION_COL='{ATTRACTION_COL}'.\n"
        f"当前字段包括：{list(all_df.columns)}\n"
        f"如果你的景点字段不是 park_name，请把 ATTRACTION_COL 改成真实字段名。"
    )

print("\n最终参与聚类图片数:", len(all_df))
print("最终 X shape:", X.shape)
print("景点数量:", all_df[ATTRACTION_COL].nunique())

all_df.to_csv(
    OUT_DIR / "all_sources_merged_manifest_valid_rebuilt.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 4. PCA 降维
# =========================

pca_dim = min(50, X.shape[0] - 1, X.shape[1])

print("\n开始 PCA...")
pca = PCA(n_components=pca_dim, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

print("PCA shape:", X_pca.shape)
print("PCA explained variance sum:", round(float(pca.explained_variance_ratio_.sum()), 4))

np.save(OUT_DIR / "X_pca_50_rebuilt.npy", X_pca)

# 释放原始大矩阵，省内存
del X
gc.collect()

# =========================
# 5. UMAP 2D 可视化坐标
# =========================

print("\n开始 UMAP...")
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)

X_umap = umap_model.fit_transform(X_pca)

all_df["umap_x"] = X_umap[:, 0]
all_df["umap_y"] = X_umap[:, 1]

np.save(OUT_DIR / "X_umap_2d_rebuilt.npy", X_umap)

print("UMAP shape:", X_umap.shape)

# =========================
# 6. 图片拼图函数
# =========================

def make_image_grid(paths, out_path, thumb_size=160, cols=6, max_images=36):
    paths = list(paths)[:max_images]
    imgs = []

    for p in paths:
        try:
            if not isinstance(p, str):
                continue
            if not os.path.exists(p):
                continue

            img = Image.open(p).convert("RGB")
            img = ImageOps.exif_transpose(img)
            img.thumbnail((thumb_size, thumb_size))

            canvas = Image.new("RGB", (thumb_size, thumb_size), (255, 255, 255))
            x = (thumb_size - img.width) // 2
            y = (thumb_size - img.height) // 2
            canvas.paste(img, (x, y))
            imgs.append(canvas)

        except Exception:
            continue

    if len(imgs) == 0:
        return False

    rows = math.ceil(len(imgs) / cols)
    grid = Image.new("RGB", (cols * thumb_size, rows * thumb_size), (255, 255, 255))

    for i, img in enumerate(imgs):
        x = (i % cols) * thumb_size
        y = (i // cols) * thumb_size
        grid.paste(img, (x, y))

    grid.save(out_path)
    return True

# =========================
# 7. 景点排序
# =========================

attraction_order = (
    all_df.groupby(ATTRACTION_COL)
    .size()
    .sort_values(ascending=False)
    .index
)

# =========================
# 8. 循环跑 K 值
# =========================

for K_FINAL in K_CANDIDATES:
    print("\n" + "=" * 70)
    print(f"Running final KMeans with K = {K_FINAL}")
    print("=" * 70)

    label_col = f"kmeans_k{K_FINAL}"

    km = MiniBatchKMeans(
        n_clusters=K_FINAL,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init="auto",
        max_iter=500
    )

    all_df[label_col] = km.fit_predict(X_pca)

    # ------------------------------------------------------------
    # 8.1 保存 image-level 聚类结果
    # ------------------------------------------------------------
    out_csv = OUT_DIR / f"all_image_level_joint_clusters_k{K_FINAL}.csv"
    all_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("Saved image-level CSV:", out_csv)

    # ------------------------------------------------------------
    # 8.2 景点 × cluster 分布表
    # ------------------------------------------------------------
    count_df = (
        all_df.groupby([ATTRACTION_COL, label_col])
        .size()
        .rename("count")
        .reset_index()
    )

    total_df = (
        all_df.groupby(ATTRACTION_COL)
        .size()
        .rename("total_images")
        .reset_index()
    )

    count_df = count_df.merge(total_df, on=ATTRACTION_COL, how="left")
    count_df["ratio"] = count_df["count"] / count_df["total_images"]

    count_wide = count_df.pivot(
        index=ATTRACTION_COL,
        columns=label_col,
        values="count"
    ).fillna(0).astype(int)

    ratio_wide = count_df.pivot(
        index=ATTRACTION_COL,
        columns=label_col,
        values="ratio"
    ).fillna(0)

    summary = ratio_wide.copy()
    summary["dominant_cluster"] = ratio_wide.idxmax(axis=1)
    summary["dominant_ratio"] = ratio_wide.max(axis=1)
    summary["total_images"] = total_df.set_index(ATTRACTION_COL)["total_images"]

    count_df.to_csv(
        OUT_DIR / f"attraction_kmeans_k{K_FINAL}_distribution_long.csv",
        index=False,
        encoding="utf-8-sig"
    )

    count_wide.to_csv(
        OUT_DIR / f"attraction_kmeans_k{K_FINAL}_count_wide.csv",
        encoding="utf-8-sig"
    )

    ratio_wide.to_csv(
        OUT_DIR / f"attraction_kmeans_k{K_FINAL}_ratio_wide.csv",
        encoding="utf-8-sig"
    )

    summary.to_csv(
        OUT_DIR / f"attraction_kmeans_k{K_FINAL}_summary.csv",
        encoding="utf-8-sig"
    )

    print("Saved attraction distribution tables.")

    # ------------------------------------------------------------
    # 8.3 每个 cluster 的代表图片拼图
    # ------------------------------------------------------------
    sample_dir_k = OUT_DIR / f"cluster_sample_grids_k{K_FINAL}"
    sample_dir_k.mkdir(exist_ok=True)

    if "abs_path" in all_df.columns:
        for c in sorted(all_df[label_col].unique()):
            sub = all_df[all_df[label_col] == c].copy()

            # 过滤掉不存在或路径无效的图片
            sub = sub[
                sub["abs_path"].apply(
                    lambda x: isinstance(x, str) and os.path.exists(x)
                )
            ]

            if len(sub) == 0:
                print(f"K={K_FINAL}, Cluster {c}: 没有找到可读取图片，跳过。")
                continue

            sample_n = min(36, len(sub))
            sample_sub = sub.sample(sample_n, random_state=RANDOM_STATE)

            out_img = sample_dir_k / f"kmeans_k{K_FINAL}_cluster_{c}_sample_grid.jpg"
            ok = make_image_grid(sample_sub["abs_path"], out_img)

            if ok:
                print(f"K={K_FINAL}, Cluster {c}: n={len(sub):,}, 已保存 -> {out_img}")
            else:
                print(f"K={K_FINAL}, Cluster {c}: 拼图失败。")
    else:
        print("没有 abs_path 字段，跳过代表图片拼图。")

    # ------------------------------------------------------------
    # 8.4 UMAP 聚类图
    # ------------------------------------------------------------
    fig_dir_k = OUT_DIR / f"figures_k{K_FINAL}"
    fig_dir_k.mkdir(exist_ok=True)

    plt.figure(figsize=(10, 8))
    plt.scatter(
        all_df["umap_x"],
        all_df["umap_y"],
        c=all_df[label_col],
        s=2,
        alpha=0.6
    )
    plt.title(f"Joint DINOv3 Image Clustering: KMeans k={K_FINAL}")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.tight_layout()

    umap_png = fig_dir_k / f"umap_kmeans_k{K_FINAL}.png"
    plt.savefig(umap_png, dpi=300)
    plt.show()

    print("Saved UMAP figure:", umap_png)

    # ------------------------------------------------------------
    # 8.5 景点 × cluster 比例热图
    # ------------------------------------------------------------
    plot_ratio = ratio_wide.reindex(attraction_order).fillna(0)

    plt.figure(figsize=(12, max(6, 0.28 * len(plot_ratio))))
    plt.imshow(plot_ratio.values, aspect="auto")
    plt.colorbar(label="Cluster ratio")

    plt.yticks(
        range(len(plot_ratio.index)),
        plot_ratio.index,
        fontsize=8  # Added fontsize for better readability
    )
    plt.xticks(
        range(len(plot_ratio.columns)),
        [f"C{c}" for c in plot_ratio.columns],
        fontsize=8  # Added fontsize for better readability
    )

    plt.xlabel("KMeans cluster")
    plt.ylabel("Attraction / scenic spot")
    plt.title(f"Cluster composition by attraction, K={K_FINAL}")
    plt.tight_layout()

    heatmap_png = fig_dir_k / f"attraction_kmeans_k{K_FINAL}_ratio_heatmap.png"
    plt.savefig(heatmap_png, dpi=300)
    plt.show()

    print("Saved heatmap:", heatmap_png)

    # ------------------------------------------------------------
    # 8.6 每个 cluster 最集中的景点
    # ------------------------------------------------------------
    cluster_top_rows = []

    for c in sorted(all_df[label_col].unique()):
        sub_c = all_df[all_df[label_col] == c]
        cluster_total = len(sub_c)

        top_attractions = (
            sub_c.groupby(ATTRACTION_COL)
            .size()
            .sort_values(ascending=False)
            .head(10)
        )

        for attraction, n in top_attractions.items():
            cluster_top_rows.append({
                "cluster": c,
                "attraction": attraction,
                "count": int(n),
                "ratio_within_cluster": float(n / cluster_total),
                "cluster_total": int(cluster_total)
            })

    cluster_top_df = pd.DataFrame(cluster_top_rows)

    cluster_top_df.to_csv(
        OUT_DIR / f"kmeans_k{K_FINAL}_cluster_top_attractions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved cluster top attractions table.")

print("\n全部完成。主要输出位置：")
for K_FINAL in K_CANDIDATES:
    print(f"OUT_DIR for K={K_FINAL} =", OUT_DIR)

print("\n代表图片文件夹：")
for K_FINAL in K_CANDIDATES:
    print(" -", OUT_DIR / f"cluster_sample_grids_k{K_FINAL}")

print("\n景点分布比例表：")
for K_FINAL in K_CANDIDATES:
    print(" -", OUT_DIR / f"attraction_kmeans_k{K_FINAL}_ratio_wide.csv")

print("\n每个景点主导 cluster：")
for K_FINAL in K_CANDIDATES:
    print(" -", OUT_DIR / f"attraction_kmeans_k{K_FINAL}_summary.csv")


In [ ]:
from pathlib import Path
import os

# 明确设置 IMAGE_ROOT_DIR 为最新修正的路径，以确保验证的是正确的路径
IMAGE_ROOT_DIR = Path("/content/drive/MyDrive/CES分析/图片")

print(f"正在验证 IMAGE_ROOT_DIR: {IMAGE_ROOT_DIR}")

if IMAGE_ROOT_DIR.exists():
    print("✅ IMAGE_ROOT_DIR 存在！")
    try:
        print(f"以下是目录 '{IMAGE_ROOT_DIR}' 中的前20个文件或文件夹：")
        items = os.listdir(IMAGE_ROOT_DIR)
        if not items:
            print("  (目录为空)")
        else:
            for i, item in enumerate(items):
                if i >= 20:
                    print("  ...")
                    break
                print(f"  - {item}")
    except Exception as e:
        print(f"❌ 无法列出目录内容：{e}")
else:
    print(f"❌ 错误：IMAGE_ROOT_DIR 目录不存在: {IMAGE_ROOT_DIR}")
    print("请检查 Google Drive 是否已正确挂载，以及路径是否拼写正确。")

print("\n--- IMAGE_ROOT_DIR 路径验证完成 ---")


In [ ]:
import os
from pathlib import Path
import pandas as pd

print("\n--- 诊断修正后的图片路径 --- ")

# 确保 IMAGE_ROOT_DIR 变量已在之前的单元格中定义
# 如果 IMAGE_ROOT_DIR 不存在，则使用XQPGSYjnibUq中的默认值
if 'IMAGE_ROOT_DIR' not in locals():
    IMAGE_ROOT_DIR = Path("/content/drive/MyDrive/CES分析/图片")

print(f"当前使用的图片文件根目录 (IMAGE_ROOT_DIR): {IMAGE_ROOT_DIR}\n")

if 'all_df' not in locals() or all_df.empty:
    print("all_df DataFrame 不存在或为空，无法进行路径验证。请确保运行了上方的处理单元格。")
elif 'abs_path' not in all_df.columns:
    print("all_df 中没有 'abs_path' 列，无法验证图片路径。")
else:
    # 随机抽取一些路径进行验证
    # 确保抽取数量不超过 all_df 的总行数
    sample_count = min(10, len(all_df))
    if sample_count == 0:
        print("all_df 中没有数据可供验证。")
    else:
        sample_rows = all_df.dropna(subset=['rel_path']).sample(sample_count, random_state=42)

        print(f"将随机抽取 {len(sample_rows)} 条 'rel_path' 对应的 'abs_path' 进行检查：")
        for i, (idx, row) in enumerate(sample_rows.iterrows()):
            rel_path = row['rel_path']
            # 重新构造路径，确保使用的是 IMAGE_ROOT_DIR
            constructed_abs_path = IMAGE_ROOT_DIR / Path(rel_path)
            existing_abs_path = Path(row['abs_path'])

            print(f"\n样本 {i+1}:")
            print(f"  原始 rel_path: {rel_path}")
            print(f"  理论构造 abs_path (IMAGE_ROOT_DIR / rel_path): {constructed_abs_path}")
            print(f"  all_df 中记录的 abs_path: {existing_abs_path}")

            if constructed_abs_path == existing_abs_path:
                print("  理论构造路径与 all_df 中记录的路径一致。")
            else:
                print("  !!! 警告: 理论构造路径与 all_df 中记录的路径不一致。")

            if existing_abs_path.exists():
                print(f"  文件存在: {existing_abs_path.exists()}")
                try:
                    # 尝试打开文件，确保它是有效的图片
                    from PIL import Image
                    Image.open(existing_abs_path)
                    print("  成功打开为图片 (PIL)。")
                except Exception as e:
                    print(f"  警告: 路径存在但无法打开为图片: {e}")
            else:
                print(f"  文件存在: {existing_abs_path.exists()} (!!! 文件不存在 !!!)")
                parent_dir = existing_abs_path.parent
                print(f"  父目录: {parent_dir}")
                if parent_dir.is_dir():
                    print("  父目录存在，列出其中内容 (最多前10个文件/文件夹，不递归):")
                    try:
                        items_in_parent = os.listdir(parent_dir)
                        if not items_in_parent:
                            print("    (父目录为空)")
                        else:
                            for j, item in enumerate(items_in_parent):
                                if j >= 10:
                                    print("    ...")
                                    break
                                print(f"    - {item}")
                    except Exception as e:
                        print(f"  无法列出父目录内容: {e}")
                else:
                    print(f"  父目录不存在: {parent_dir} (!!! 这是一个问题 !!!)")

print("\n--- 路径诊断完成 ---\n请仔细检查 '文件存在: False' 的条目，以及父目录下的文件列表，确认 IMAGE_ROOT_DIR 和 rel_path 的组合是否能正确指向 Google Drive 中的图片文件。这通常意味着 IMAGE_ROOT_DIR 或 manifest.csv 中的 rel_path 需要进一步修正。")

In [ ]:
import os
from pathlib import Path

# 确保 IMAGE_ROOT_DIR 变量已定义
# 如果没有，则使用预设的路径
if 'IMAGE_ROOT_DIR' not in locals():
    IMAGE_ROOT_DIR = Path("/content/drive/MyDrive/CES分析/图片 ")

print(f"正在列出 IMAGE_ROOT_DIR ({IMAGE_ROOT_DIR}) 的直接内容：")

if IMAGE_ROOT_DIR.exists():
    try:
        items = os.listdir(IMAGE_ROOT_DIR)
        if not items:
            print("  (目录为空)")
        else:
            # 打印前20个项目，以避免输出过长
            for i, item in enumerate(items):
                if i >= 20:
                    print("  ...")
                    break
                print(f"  - {item}")
    except Exception as e:
        print(f"  无法列出目录内容: {e}")
else:
    print(f"  !! 错误：IMAGE_ROOT_DIR 目录不存在: {IMAGE_ROOT_DIR}\n请检查 Google Drive 挂载是否正确，或者路径是否拼写错误。")

print("\n请根据上述输出，确认 '一亩园公园', '颐和园' 等公园名称是否作为直接子目录存在于 IMAGE_ROOT_DIR 中，或者它们是否嵌套在其他目录之下。")

### 检查 SOURCE_DIRS 中的文件

请运行以下代码，检查每个 `SOURCE_DIRS` 路径下是否包含 `manifest.csv` 和 `features.h5` 文件。如果文件不存在，请确保它们已上传到 Google Drive 的正确位置。

In [ ]:
from pathlib import Path
import os

# =========================
# 四批 DINOv3 编码结果目录
# (Copied from cell 22Y7ZXJJTdUt to ensure it's defined after runtime restarts)
# =========================
SOURCE_DIRS = {
    "all_parks": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid"),
    "xiangshan_all": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_xs_all"),
    "yuanmingyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_ymy"),
    "yiheyuan": Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid_yhy"),
}

for tag, path in SOURCE_DIRS.items():
    print(f"\nChecking directory for '{tag}': {path}")
    if path.exists():
        print("  Directory exists.")
        manifest_path = path / "manifest.csv"
        features_path = path / "features.h5"
        print(f"  manifest.csv exists: {manifest_path.exists()}")
        print(f"  features.h5 exists: {features_path.exists()}")
        if not manifest_path.exists() or not features_path.exists():
            print(f"  !! WARNING: Either manifest.csv or features.h5 is missing in {path}")
        print("  Listing first 10 files/directories in this path:")
        try:
            for i, item in enumerate(os.listdir(path)):
                if i >= 10:
                    print("  ...")
                    break
                print(f"  - {item}")
        except Exception as e:
            print(f"  Error listing directory contents: {e}")
    else:
        print(f"  !! ERROR: Directory does not exist: {path}")


如果上述检查显示文件确实不存在，你需要将它们上传到 Google Drive 的对应路径。如果路径错误，你需要修改 `SOURCE_DIRS` 字典中的路径。

关于 `ValueError: Mountpoint must not already contain files`，这是一个常见的 Colab 问题。这意味着在你尝试挂载 Google Drive 时，`/content/drive` 目录不是空的。如果再次遇到这个错误，可以尝试运行以下命令来清理或重新挂载您的云端硬盘：

```python
!fusermount -uz /content/drive
!rm -rf /content/drive/*
from google.colab import drive
drive.mount('/content/drive')
```

**请注意**：`!rm -rf /content/drive/*` 会删除 `/content/drive` 目录下的所有内容。如果你的 Google Drive 之前已经成功挂载并创建了文件在 `/content/drive` 里面，这可能会删除它们。但通常情况下，`/content/drive` 应该是一个空的挂载点，所以这个命令是安全的。

In [ ]:
from pathlib import Path
import os

# 检查 SOURCE_DIRS 中的第一个路径
first_source_dir_tag = list(SOURCE_DIRS.keys())[0]
first_source_path = SOURCE_DIRS[first_source_dir_tag]

print(f"\n正在验证第一个源目录 '{first_source_dir_tag}': {first_source_path}")

if first_source_path.exists():
    print(f"  目录 '{first_source_path}' 存在。")
    print("  列出该目录下的前10个文件/文件夹，以确认内容：")
    try:
        items = os.listdir(first_source_path)
        if not items:
            print("  (目录为空)")
        else:
            for i, item in enumerate(items):
                if i >= 10:
                    print("  ...")
                    break
                print(f"  - {item}")
        manifest_check = first_source_path / "manifest.csv"
        features_check = first_source_path / "features.h5"
        print(f"  'manifest.csv' 是否存在: {manifest_check.exists()}")
        print(f"  'features.h5' 是否存在: {features_check.exists()}")
    except Exception as e:
        print(f"  无法列出目录内容或检查文件: {e}")
else:
    print(f"  !! 错误：目录 '{first_source_path}' 不存在。请检查路径或 Google Drive 挂载。")

## 8. 设置最终聚类数

如果自动选择的 `best_k` 不好解释，可以手动指定，例如 `K_FINAL = 6` 或 `K_FINAL = 8`。

In [ ]:
# 自动选择
# K_FINAL = best_k

# 如需手动指定，取消下面这一行注释：
# K_FINAL = 8

# print("最终使用 K =", K_FINAL)

In [ ]:
K_FINAL = 10
print("最终使用 K =", K_FINAL)

## 加载并展示 K=10 的结果

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 确保 OUT_DIR 与 cell XQPGSYjnibUq 中设置的路径一致
OUT_DIR = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv生成数据2")
FIG_DIR = OUT_DIR / f"figures_k{K_FINAL}"
SAMPLE_DIR = OUT_DIR / f"cluster_sample_grids_k{K_FINAL}"
ATTRACTION_COL = "park_name"

# 1. 加载图片级聚类结果
image_level_csv_path = OUT_DIR / f"all_image_level_joint_clusters_k{K_FINAL}.csv"
all_df = pd.read_csv(image_level_csv_path)
print(f"已加载图片级聚类结果: {image_level_csv_path}")
display(all_df.head())

# 2. 加载景点与聚类分布总结表
summary_csv_path = OUT_DIR / f"attraction_kmeans_k{K_FINAL}_summary.csv"
km_summary = pd.read_csv(summary_csv_path)
print(f"\n已加载景点与聚类分布总结表: {summary_csv_path}")
display(km_summary.head())

In [ ]:
import openpyxl # 确保 openpyxl 库已加载，用于支持 .xlsx 格式输出

# K_FINAL 变量应已在之前的单元格中定义
# 构建动态的 all_df CSV 路径
new_image_level_csv_path = OUT_DIR / f"all_image_level_joint_clusters_k{K_FINAL}.csv"

# 重新定义 make_distribution_tables 函数，确保其在当前会话中可用
def make_distribution_tables(df, group_col, cluster_col, out_prefix, output_dir):
    count_df = (
        df.groupby([group_col, cluster_col])
        .size()
        .rename("count")
        .reset_index()
    )

    total_df = (
        df.groupby(group_col)
        .size()
        .rename("total_images")
        .reset_index()
    )

    count_df = count_df.merge(total_df, on=group_col, how="left")
    count_df["ratio"] = count_df["count"] / count_df["total_images"]

    count_wide = count_df.pivot(
        index=group_col,
        columns=cluster_col,
        values="count"
    ).fillna(0).astype(int)

    ratio_wide = count_df.pivot(
        index=group_col,
        columns=cluster_col,
        values="ratio"
    ).fillna(0)

    summary = ratio_wide.copy()
    summary["dominant_cluster"] = ratio_wide.idxmax(axis=1)
    summary["dominant_ratio"] = ratio_wide.max(axis=1)
    summary["total_images"] = total_df.set_index(group_col)["total_images"]

    # Save to XLSX instead of CSV
    count_df.to_excel(output_dir / f"{out_prefix}_long.xlsx", index=False)
    count_wide.to_excel(output_dir / f"{out_prefix}_count_wide.xlsx")
    ratio_wide.to_excel(output_dir / f"{out_prefix}_ratio_wide.xlsx")
    summary.to_excel(output_dir / f"{out_prefix}_summary.xlsx")

    return count_df, count_wide, ratio_wide, summary


if new_image_level_csv_path.exists():
    print(f"从 '{new_image_level_csv_path}' 加载 all_df...")
    try:
        # 尝试加载新文件
        temp_all_df = pd.read_csv(new_image_level_csv_path)
        label_col = f"kmeans_k{K_FINAL}"

        if label_col in temp_all_df.columns:
            print(f"成功加载文件，并且找到了 '{label_col}' 列。现在使用这个 DataFrame 作为 all_df。")
            all_df = temp_all_df.copy() # 更新 all_df

            # --- BEGIN ABS_PATH RE-CORRECTION ---
            print("重新修正 'abs_path' 列以匹配 Google Drive 图片根目录...")
            # 确保 IMAGE_ROOT_DIR 已定义 (通常在 XQPGSYjnibUq 中定义)
            if 'IMAGE_ROOT_DIR' not in globals():
                IMAGE_ROOT_DIR = Path("/content/drive/MyDrive/CES分析/图片") # Fallback if not defined

            def build_correct_image_path(row):
                if pd.isna(row["rel_path"]):
                    return None

                rel_path_obj = Path(row["rel_path"])
                # The first component of rel_path is typically the park name (e.g., '颐和园', '一亩园公园')
                first_component_of_rel_path = rel_path_obj.parts[0]

                # Determine the intermediate folder name based on the first component
                intermediate_dir_map = {
                    "颐和园": "图片下载-颐和园",
                    "圆明园": "图片下载-圆明园",
                    "香山": "图片下载-香山-植物园",
                    "香山-植物园": "图片下载-香山-植物园",
                    "植物园": "图片下载-香山-植物园" # If rel_path starts with just '植物园'
                }

                # Default for other parks not explicitly mapped
                intermediate_dir = intermediate_dir_map.get(first_component_of_rel_path, "图片下载-公园")

                # Construct the full absolute path: IMAGE_ROOT_DIR / IntermediateDir / RelPath
                return str(IMAGE_ROOT_DIR / intermediate_dir / rel_path_obj)

            if "rel_path" in all_df.columns:
                all_df["abs_path"] = all_df.apply(build_correct_image_path, axis=1)
                print("'abs_path' 列修正完成。")
            else:
                print("警告: all_df 中没有 'rel_path' 列，跳过 'abs_path' 修正。")
            # --- END ABS_PATH RE-CORRECTION ---

            # 重新执行之前的清理、聚合和分布计算
            print('\n--- 重新处理 all_df 并计算景点聚类分布 ---')

            # 注意: ATTRACTION_COL, OUT_DIR, K_FINAL 应该已经在之前的单元格中定义

            print(f"清理前，景点数量: {all_df[ATTRACTION_COL].nunique()}")

            # 1. 景点名称清理与统一 (同 eda83655)
            all_df[ATTRACTION_COL] = all_df[ATTRACTION_COL].astype(str)
            all_df[ATTRACTION_COL] = (
                all_df[ATTRACTION_COL]
                .str.lower()
                .str.replace('_', ' ')
                .str.replace(r'\\s+', ' ', regex=True) # 替换多个空格为单个空格
                .str.strip() # 去除首尾空格
            )
            print(f"清理后，景点数量: {all_df[ATTRACTION_COL].nunique()}")

            # 2. 景点层级统一化 (同 904aac44)
            major_park_aggregation_map = {
                "xiangshan_all": "香山-植物园",
                "yuanmingyuan": "圆明园",
                "yiheyuan": "颐和园",
            }
            for source_tag, major_park_name in major_park_aggregation_map.items():
                # 检查 source_tag 是否存在于 all_df 中
                if "source_tag" in all_df.columns:
                    all_df.loc[all_df["source_tag"] == source_tag, ATTRACTION_COL] = major_park_name
                else:
                    print("警告: all_df 中没有 'source_tag' 列，跳过景点层级统一化部分。")
                    break # 如果没有 source_tag，就没法按 source_tag 统一
            print(f"层级统一化后，景点数量: {all_df[ATTRACTION_COL].nunique()}")

            # 3. 过滤掉 '百旺绿地公园' (同 f4b46a51)
            exclude_park_name = "百旺绿地公园"
            keep_mask = all_df[ATTRACTION_COL] != exclude_park_name
            all_df = all_df[keep_mask].reset_index(drop=True)
            print(f"过滤 '{exclude_park_name}' 后，图片数量: {len(all_df):,}")
            print(f"最终景点数量: {all_df[ATTRACTION_COL].nunique()}")

            # 4. 再次计算景点聚类分布
            # make_distribution_tables 函数已在之前的 `4ca42aa2` 单元格中定义
            # 确保 OUT_DIR 变量已在 notebook 中定义
            km_long, km_count_wide, km_ratio_wide, km_summary = make_distribution_tables(
                all_df,
                ATTRACTION_COL,
                label_col,
                f"attraction_kmeans_k{K_FINAL}_distribution_user_file_processed", # 使用新前缀
                OUT_DIR
            )

            print(f"\n每个景点的 K={K_FINAL} 聚类占比已计算并保存到 {OUT_DIR} 目录。")
            print("显示最终景点列表和聚类分布总结 (km_summary) 的前几行：")
            display(km_summary.head())
            print("显示最终景点列表和聚类比例宽格式 (km_ratio_wide) 的前几行：")
            display(km_ratio_wide.head())
            print("所有独特的公园名称：")
            display(all_df[ATTRACTION_COL].unique())

        else:
            print(f"错误: 新加载的 all_df 中未找到列 '{label_col}'。请检查文件内容。")
            print("当前新加载 all_df 的列有:", temp_all_df.columns.tolist())
    except Exception as e:
        print(f"加载或处理新文件时发生错误: {e}")
else:
    print(f"错误: 用户指定的文件不存在: '{new_image_level_csv_path}'")

### K=10 UMAP 聚类图

In [ ]:
umap_png_path = FIG_DIR / f"umap_kmeans_k{K_FINAL}.png"
if umap_png_path.exists():
    print(f"加载并显示 UMAP 聚类图: {umap_png_path}")
    img = Image.open(umap_png_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Joint DINOv3 Image Clustering: KMeans k={K_FINAL}")
    plt.show()
else:
    print(f"警告: UMAP 聚类图文件不存在: {umap_png_path}")

### K=10 景点 × 聚类比例热图

In [ ]:
heatmap_png_path = FIG_DIR / f"attraction_kmeans_k{K_FINAL}_ratio_heatmap.png"
if heatmap_png_path.exists():
    print(f"加载并显示景点 × 聚类比例热图: {heatmap_png_path}")
    img = Image.open(heatmap_png_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Cluster composition by attraction, K={K_FINAL}")
    plt.show()
else:
    print(f"警告: 景点 × 聚类比例热图文件不存在: {heatmap_png_path}")

### K=10 每个 cluster 最集中的景点

In [ ]:
cluster_top_attractions_path = OUT_DIR / f"kmeans_k{K_FINAL}_cluster_top_attractions.csv"
cluster_top_df = pd.read_csv(cluster_top_attractions_path)
print(f"已加载每个 cluster 最集中的景点表: {cluster_top_attractions_path}")
display(cluster_top_df.head(10))

### K=10 代表图片文件夹路径

In [ ]:
print(f"K={K_FINAL} 的代表图片拼图文件夹: {SAMPLE_DIR}")
# 你可以手动访问此路径下的图片文件查看每个聚类的代表图像。

### 为 K=10 的每个聚类生成代表性图片拼图

请在确认 Google Drive 已成功挂载后运行以下单元格。它将使用之前加载的 `all_df` 数据和 `K_FINAL` 设置来生成每个聚类的代表性图片拼图。

In [ ]:
from PIL import Image, ImageOps
import math
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 定义 make_image_grid 函数，以防在重启后丢失定义
def make_image_grid(paths, out_path, thumb_size=160, cols=6, max_images=36):
    paths = list(paths)[:max_images]
    imgs = []

    for p in paths:
        try:
            if not isinstance(p, str):
                continue
            if not os.path.exists(p):
                continue

            img = Image.open(p).convert("RGB")
            img = ImageOps.exif_transpose(img)
            img.thumbnail((thumb_size, thumb_size))

            canvas = Image.new("RGB", (thumb_size, thumb_size), (255, 255, 255))
            x = (thumb_size - img.width) // 2
            y = (thumb_size - img.height) // 2
            canvas.paste(img, (x, y))
            imgs.append(canvas)

        except Exception as e:
            print(f"处理图片 {p} 时发生错误: {e}")
            continue

    if len(imgs) == 0:
        return False

    rows = math.ceil(len(imgs) / cols)
    grid = Image.new("RGB", (cols * thumb_size, rows * thumb_size), (255, 255, 255))

    for i, img in enumerate(imgs):
        x = (i % cols) * thumb_size
        y = (i // cols) * thumb_size
        grid.paste(img, (x, y))

    grid.save(out_path)
    return True

print(f"开始为 K={K_FINAL} 生成每个聚类的代表图片拼图...")

# K_FINAL, OUT_DIR, all_df, RANDOM_STATE 应该已经从之前的单元格中定义
label_col = f"kmeans_k{K_FINAL}"
sample_dir_k = OUT_DIR / f"cluster_sample_grids_k{K_FINAL}"
sample_dir_k.mkdir(exist_ok=True)

if "abs_path" in all_df.columns:
    for c in sorted(all_df[label_col].unique()):
        sub = all_df[all_df[label_col] == c].copy()

        # 过滤掉不存在或路径无效的图片
        sub = sub[
            sub["abs_path"].apply(
                lambda x: isinstance(x, str) and os.path.exists(x)
            )
        ]

        if len(sub) == 0:
            print(f"K={K_FINAL}, Cluster {c}: 没有找到可读取图片，跳过。")
            continue

        sample_n = min(36, len(sub))
        sample_sub = sub.sample(sample_n, random_state=RANDOM_STATE)

        out_img = sample_dir_k / f"kmeans_k{K_FINAL}_cluster_{c}_sample_grid.jpg"
        ok = make_image_grid(sample_sub["abs_path"], out_img)

        if ok:
            print(f"K={K_FINAL}, Cluster {c}: n={len(sub):,}, 已保存 -> {out_img}")
        else:
            print(f"K={K_FINAL}, Cluster {c}: 拼图失败。")
else:
    print("没有 abs_path 字段，跳过代表图片拼图。")

print("代表图片拼图生成完毕。请检查目录:", sample_dir_k)


### 诊断 Google Drive 文件夹结构

以下单元格将列出 `SOURCE_DIRS` 中第一个目录（通常是 `DINOv3_park_analysis_uid`）下的文件夹和文件结构，帮助您检查实际的图片存放位置是否与 `abs_path` 列中记录的路径相符。我们将递归地列出子目录的内容，但会限制深度以避免输出过长。

In [ ]:
from pathlib import Path
import os

def list_directory_structure(path, indent=0, max_depth=3, current_depth=0):
    if current_depth > max_depth:
        return

    try:
        for item in os.listdir(path):
            item_path = path / item
            print('  ' * indent + f'- {item_path.name}')
            if item_path.is_dir():
                list_directory_structure(item_path, indent + 1, max_depth, current_depth + 1)
    except Exception as e:
        print('  ' * indent + f'Error accessing {path.name}: {e}')

print("正在列出第一个 SOURCE_DIR 的内容...\n")
first_source_path = list(SOURCE_DIRS.values())[0]
print(f"检查目录: {first_source_path}\n")

if first_source_path.exists():
    list_directory_structure(first_source_path)
else:
    print(f"错误: 目录 {first_source_path} 不存在。请确认 Google Drive 已正确挂载且路径正确。")

print("\n请根据上述输出检查您的 Google Drive 中图片的实际存放路径，与 `all_df` 中 `abs_path` 列的路径进行比对，必要时手动调整 Google Drive 的文件结构。")

In [ ]:
import os
from pathlib import Path
import pandas as pd

print("\n--- 验证 all_df 中的图片路径 --- ")

# 假设 all_df 和 ATTRACTION_COL 已经定义
if 'abs_path' not in all_df.columns:
    print("all_df 中没有 'abs_path' 列，无法验证图片路径。")
else:
    # 随机抽取一些路径进行验证
    sample_paths = all_df['abs_path'].dropna().sample(min(10, len(all_df)), random_state=RANDOM_STATE)

    if len(sample_paths) == 0:
        print("all_df 中没有有效的图片路径可供验证。")
    else:
        print(f"将随机抽取 {len(sample_paths)} 条 'abs_path' 进行检查：")
        for i, p_str in enumerate(sample_paths):
            print(f"\n样本 {i+1}: {p_str}")
            p = Path(p_str)
            if p.exists():
                print(f"  文件存在: {p.exists()}")
                try:
                    # 尝试打开文件，确保它是有效的图片
                    Image.open(p)
                    print("  成功打开为图片。")
                except Exception as e:
                    print(f"  警告: 路径存在但无法打开为图片: {e}")
            else:
                print(f"  文件存在: {p.exists()} (!!! 文件不存在 !!!)")
                parent_dir = p.parent
                print(f"  父目录: {parent_dir}")
                if parent_dir.is_dir():
                    print("  父目录存在，列出其中内容 (最多前10个):")
                    try:
                        items_in_parent = os.listdir(parent_dir)
                        if not items_in_parent:
                            print("    (父目录为空)")
                        else:
                            for j, item in enumerate(items_in_parent):
                                if j >= 10:
                                    print("    ...")
                                    break
                                print(f"    - {item}")
                    except Exception as e:
                        print(f"  无法列出父目录内容: {e}")
                else:
                    print(f"  父目录不存在: {parent_dir}")

print("\n--- 验证完成 ---\n请仔细检查 '文件存在: False' 的条目，以及其父目录下的文件列表，确认图片文件是否确实存在于 Google Drive 的对应位置。" )

## K=10 的主要输出文件清单

In [ ]:
print(f"输出目录: {OUT_DIR}")
print(f"1. 图片级聚类结果: {OUT_DIR}/all_image_level_joint_clusters_k{K_FINAL}.csv")
print(f"2. 景点×KMeans 聚类比例 (宽格式): {OUT_DIR}/attraction_kmeans_k{K_FINAL}_ratio_wide.csv")
print(f"3. 景点×KMeans 聚类数量 (宽格式): {OUT_DIR}/attraction_kmeans_k{K_FINAL}_count_wide.csv")
print(f"4. 每个景点主导视觉类型总结: {OUT_DIR}/attraction_kmeans_k{K_FINAL}_summary.csv")
print(f"5. 每个 cluster 集中出现的景点: {OUT_DIR}/kmeans_k{K_FINAL}_cluster_top_attractions.csv")
print(f"6. K={K_FINAL} 代表图片拼图文件夹: {SAMPLE_DIR}")
print(f"7. K={K_FINAL} 图件文件夹: {FIG_DIR}")

## 9. 最终 KMeans 聚类

In [ ]:
# kmeans = MiniBatchKMeans(
#     n_clusters=K_FINAL,
#     random_state=RANDOM_STATE,
#     batch_size=4096,
#     n_init="auto",
#     max_iter=500
# )

# all_df["kmeans_cluster"] = kmeans.fit_predict(X_pca)

# print(all_df["kmeans_cluster"].value_counts().sort_index())

## 10. UMAP 二维可视化降维

UMAP 主要用于可视化，不建议直接把二维 UMAP 当作唯一聚类依据。

In [ ]:
# umap_model = umap.UMAP(
#     n_components=2,
#     n_neighbors=30,
#     min_dist=0.05,
#     metric="cosine",
#     random_state=RANDOM_STATE,
#     low_memory=True
# )

# X_umap = umap_model.fit_transform(X_pca)

# all_df["umap_x"] = X_umap[:, 0]
# all_df["umap_y"] = X_umap[:, 1]

# np.save(OUT_DIR / "X_umap_2d.npy", X_umap)

# print("UMAP shape:", X_umap.shape)

## 11. 补充：HDBSCAN 聚类

HDBSCAN 会产生 `-1`，代表噪声点。  
建议作为补充检查，不作为主结果。

In [ ]:
# hdb = hdbscan.HDBSCAN(
#     min_cluster_size=50,
#     min_samples=10,
#     metric="euclidean",
#     prediction_data=False
# )

# all_df["hdbscan_cluster"] = hdb.fit_predict(X_umap)

# print(all_df["hdbscan_cluster"].value_counts().sort_index())

## 12. 保存 image-level 聚类结果

In [ ]:
# image_level_csv = OUT_DIR / "all_image_level_joint_clusters.csv"
# all_df.to_csv(image_level_csv, index=False, encoding="utf-8-sig")

# print("已保存:", image_level_csv)
# display(all_df.head())

## 13. 按景点统计聚类分布

In [ ]:
# def make_distribution_tables(df, group_col, cluster_col, out_prefix):
#     count_df = (
#         df.groupby([group_col, cluster_col])
#         .size()
#         .rename("count")
#         .reset_index()
#     )

#     total_df = (
#         df.groupby(group_col)
#         .size()
#         .rename("total_images")
#         .reset_index()
#     )

#     count_df = count_df.merge(total_df, on=group_col, how="left")
#     count_df["ratio"] = count_df["count"] / count_df["total_images"]

#     count_wide = count_df.pivot(
#         index=group_col,
#         columns=cluster_col,
#         values="count"
#     ).fillna(0).astype(int)

#     ratio_wide = count_df.pivot(
#         index=group_col,
#         columns=cluster_col,
#         values="ratio"
#     ).fillna(0)

#     summary = ratio_wide.copy()
#     summary["dominant_cluster"] = ratio_wide.idxmax(axis=1)
#     summary["dominant_ratio"] = ratio_wide.max(axis=1)
#     summary["total_images"] = total_df.set_index(group_col)["total_images"]

#     count_df.to_csv(OUT_DIR / f"{out_prefix}_long.csv", index=False, encoding="utf-8-sig")
#     count_wide.to_csv(OUT_DIR / f"{out_prefix}_count_wide.csv", encoding="utf-8-sig")
#     ratio_wide.to_csv(OUT_DIR / f"{out_prefix}_ratio_wide.csv", encoding="utf-8-sig")
#     summary.to_csv(OUT_DIR / f"{out_prefix}_summary.csv", encoding="utf-8-sig")

#     return count_df, count_wide, ratio_wide, summary


# km_long, km_count_wide, km_ratio_wide, km_summary = make_distribution_tables(
#     all_df,
#     ATTRACTION_COL,
#     "kmeans_cluster",
#     "attraction_kmeans_distribution"
# )

# display(km_summary.sort_values("dominant_cluster").head())

## 14. 按景点统计 HDBSCAN 分布

In [ ]:
# hdb_long, hdb_count_wide, hdb_ratio_wide, hdb_summary = make_distribution_tables(
#     all_df,
#     ATTRACTION_COL,
#     "hdbscan_cluster",
#     "attraction_hdbscan_distribution"
# )

# display(hdb_summary.head())

## 15. 绘制 UMAP 聚类图

In [ ]:
# plt.figure(figsize=(10, 8))
# plt.scatter(
#     all_df["umap_x"],
#     all_df["umap_y"],
#     c=all_df["kmeans_cluster"],
#     s=2,
#     alpha=0.6
# )
# plt.title(f"Joint DINOv3 Image Clustering: KMeans k={K_FINAL}")
# plt.xlabel("UMAP-1")
# plt.ylabel("UMAP-2")
# plt.tight_layout()
# plt.savefig(FIG_DIR / "umap_kmeans_clusters.png", dpi=300)
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 8))
# plt.scatter(
#     all_df["umap_x"],
#     all_df["umap_y"],
#     c=all_df["hdbscan_cluster"],
#     s=2,
#     alpha=0.6
# )
# plt.title("Joint DINOv3 Image Clustering: HDBSCAN")
# plt.xlabel("UMAP-1")
# plt.ylabel("UMAP-2")
# plt.tight_layout()
# plt.savefig(FIG_DIR / "umap_hdbscan_clusters.png", dpi=300)
# plt.show()

## 16. 绘制“景点 × 聚类”的比例热图

In [ ]:
# # 按图片数量排序，避免图太乱
# attraction_order = (
#     all_df.groupby(ATTRACTION_COL)
#     .size()
#     .sort_values(ascending=False)
#     .index
# )

# plot_ratio = km_ratio_wide.loc[attraction_order]

# plt.figure(figsize=(12, max(6, 0.28 * len(plot_ratio))))
# plt.imshow(plot_ratio.values, aspect="auto")
# plt.colorbar(label="Cluster ratio")

# plt.yticks(range(len(plot_ratio.index)), plot_ratio.index)
# plt.xticks(range(len(plot_ratio.columns)), [f"C{c}" for c in plot_ratio.columns])

# plt.xlabel("KMeans cluster")
# plt.ylabel("Attraction / scenic spot")
# plt.title("Cluster composition by attraction")
# plt.tight_layout()
# plt.savefig(FIG_DIR / "attraction_kmeans_ratio_heatmap.png", dpi=300)
# plt.show()

## 17. 绘制每个景点的堆叠比例图

In [ ]:
# plot_ratio = km_ratio_wide.loc[attraction_order]

# ax = plot_ratio.plot(
#     kind="bar",
#     stacked=True,
#     figsize=(max(12, 0.35 * len(plot_ratio)), 6),
#     width=0.9
# )

# ax.set_ylabel("Ratio")
# ax.set_xlabel("Attraction / scenic spot")
# ax.set_title("KMeans cluster distribution by attraction")
# ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.savefig(FIG_DIR / "attraction_kmeans_stacked_bar.png", dpi=300)
# plt.show()

## 18. 给每个 cluster 输出代表性图片拼图

这一步最重要。DINOv3 聚类本身没有语义标签，你需要根据每类代表图像手动命名。

In [ ]:
# def make_image_grid(paths, out_path, thumb_size=160, cols=6, max_images=36):
#     paths = list(paths)[:max_images]
#     imgs = []

#     for p in paths:
#         try:
#             img = Image.open(p).convert("RGB")
#             img = ImageOps.exif_transpose(img)
#             img.thumbnail((thumb_size, thumb_size))
#             canvas = Image.new("RGB", (thumb_size, thumb_size), (255, 255, 255))
#             x = (thumb_size - img.width) // 2
#             y = (thumb_size - img.height) // 2
#             canvas.paste(img, (x, y))
#             imgs.append(canvas)
#         except Exception:
#             continue

#     if len(imgs) == 0:
#         return False

#     rows = math.ceil(len(imgs) / cols)
#     grid = Image.new("RGB", (cols * thumb_size, rows * thumb_size), (255, 255, 255))

#     for i, img in enumerate(imgs):
#         x = (i % cols) * thumb_size
#         y = (i // cols) * thumb_size
#         grid.paste(img, (x, y))

#     grid.save(out_path)
#     return True


# # 每类随机抽样
# for c in sorted(all_df["kmeans_cluster"].unique()):
#     sub = all_df[all_df["kmeans_cluster"] == c].copy()

#     if "abs_path" not in sub.columns:
#         print("没有 abs_path 字段，无法输出图片拼图。")
#         break

#     sub = sub[sub["abs_path"].apply(lambda x: isinstance(x, str) and os.path.exists(x))]
#     if len(sub) == 0:
#         print(f"Cluster {c}: 没有可读取图片路径")
#         continue

#     sample_n = min(36, len(sub))
#     sample_sub = sub.sample(sample_n, random_state=RANDOM_STATE)

#     out_img = SAMPLE_DIR / f"kmeans_cluster_{c}_sample_grid.jpg"
#     ok = make_image_grid(sample_sub["abs_path"], out_img)

#     if ok:
#         print(f"Cluster {c}: n={len(sub):,}, saved -> {out_img}")

## 19. 输出每个 cluster 最集中的景点

这个表可以帮助解释每个视觉类型主要出现在哪些景点。

In [ ]:
# cluster_attraction_summary = []

# for c in sorted(all_df["kmeans_cluster"].unique()):
#     sub = all_df[all_df["kmeans_cluster"] == c]
#     total_c = len(sub)

#     top_attractions = (
#         sub.groupby(ATTRACTION_COL)
#         .size()
#         .sort_values(ascending=False)
#         .head(10)
#     )

#     for attraction, n in top_attractions.items():
#         cluster_attraction_summary.append({
#             "cluster": c,
#             "attraction": attraction,
#             "count": int(n),
#             "ratio_within_cluster": float(n / total_c),
#             "cluster_total": int(total_c)
#         })

# cluster_attraction_summary = pd.DataFrame(cluster_attraction_summary)
# cluster_top_csv = OUT_DIR / "cluster_top_attractions.csv"
# cluster_attraction_summary.to_csv(
#     cluster_top_csv,
#     index=False,
#     encoding="utf-8-sig"
# )

# print("已保存:", cluster_top_csv)
# display(cluster_attraction_summary)

## 20. 输出每个景点的主导视觉类型

In [ ]:
# dominant_table = km_summary.reset_index().rename(columns={ATTRACTION_COL: "attraction"})

# cols = ["attraction", "total_images", "dominant_cluster", "dominant_ratio"]
# other_cols = [c for c in dominant_table.columns if c not in cols]
# dominant_table = dominant_table[cols + other_cols]

# dominant_table = dominant_table.sort_values(
#     ["dominant_cluster", "dominant_ratio"],
#     ascending=[True, False]
# )

# dominant_csv = OUT_DIR / "attraction_dominant_visual_cluster.csv"
# dominant_table.to_csv(
#     dominant_csv,
#     index=False,
#     encoding="utf-8-sig"
# )

# print("已保存:", dominant_csv)
# display(dominant_table)

## 21. 输出结果清单

In [ ]:
# print("输出目录:", OUT_DIR)
# print("1. 图片级聚类结果:", OUT_DIR / "all_image_level_joint_clusters.csv")
# print("2. 景点×KMeans 聚类比例:", OUT_DIR / "attraction_kmeans_distribution_ratio_wide.csv")
# print("3. 景点×KMeans 聚类数量:", OUT_DIR / "attraction_kmeans_distribution_count_wide.csv")
# print("4. 每个景点主导视觉类型:", OUT_DIR / "attraction_dominant_visual_cluster.csv")
# print("5. 每个 cluster 集中出现的景点:", OUT_DIR / "cluster_top_attractions.csv")
# print("6. 每类代表图片拼图:", SAMPLE_DIR)
# print("7. 图件:", FIG_DIR)

## 22. 论文表述建议

主方法可以写为：

```text
We clustered image-level DINOv3 embeddings using PCA-reduced feature vectors and MiniBatch K-means. The resulting clusters were interpreted as image-derived visual framing groups rather than fixed semantic landscape categories.
```

更稳妥的中文理解是：

```text
这些聚类代表“基于图像构图与视觉相似性的视觉框架组”，不直接等同于真实景观类型或功能类型。
```

你需要根据 `cluster_sample_grids/` 里的代表图像，手动给每个 cluster 命名。例如：

```text
C0: waterside and open-view scenes
C1: dense vegetation scenes
C2: architectural / heritage-focused scenes
C3: activity and crowd scenes
C4: path and entrance scenes
C5: close-up plant or object scenes
```

实际命名必须以你生成的拼图为准。

## 模型设置描述

本 notebook 用于对四个 DINOv3 编码结果目录（`/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv数据原始/DINOv3_park_analysis_uid`, `DINOv3_park_analysis_uid_xs_all`, `DINOv3_park_analysis_uid_ymy`, `DINOv3_park_analysis_uid_yhy`）进行合并、统一聚类和景点分布分析。

处理流程如下：

1.  **数据加载与合并**：读取每个目录中的 `manifest.csv` 和 `features.h5` 文件。从 `features.h5` 中优先读取 `embeddings_l2`，如果不存在则读取 `embeddings`。对数据进行有效性检查，过滤掉无效（非有限值、全零）的 embeddings，并为每张图片生成 `global_uid`。最终合并所有来源的数据，并移除 `abs_path` 完全重复的图片，得到用于聚类的 `328,448` 张图片及其对应的 `(328448, 768)` 维特征矩阵 `X`。

2.  **PCA 降维**：为了提高聚类效率和稳定性，使用主成分分析（PCA）将 `768` 维的 DINOv3 embeddings 降维到 `50` 维 (`pca_dim = 50`)。PCA 解释的方差比约为 `0.5114`。

3.  **KMeans 聚类数选择**：为了选择合适的 `KMeans` 聚类数量 `K`，在 `K=4` 到 `K=20` 的范围内，通过计算 `silhouette score` 来评估聚类效果。`silhouette score` 的计算基于 `10,000` 张图片的子样本以加快速度。`K_CANDIDATES` 包含 `[4, 6, 8, 10, 12, 14, 16, 18, 20]`。

4.  **KMeans 聚类**：使用 `MiniBatchKMeans` 进行聚类，其参数设置为 `batch_size=4096`，`n_init="auto"`，`max_iter=500`，`random_state=42`。聚类过程对每个 `K_CANDIDATES` 中的 `K` 值单独运行。

5.  **UMAP 降维可视化**：使用 UMAP 将 `50` 维 PCA 特征进一步降维到 `2` 维，用于可视化。UMAP 参数设置为 `n_components=2`，`n_neighbors=30`，`min_dist=0.05`，`metric="cosine"`，`random_state=42`。

6.  **结果输出**：
    *   生成每个 `K` 值对应的图片级聚类结果 CSV 文件 (`all_image_level_joint_clusters_k{K}.csv`)。
    *   生成景点与聚类分布的统计表格（包括长格式、计数宽格式、比例宽格式和总结表），例如 `attraction_kmeans_k{K}_ratio_wide.csv`。
    *   为每个聚类生成代表性图片拼图 (`cluster_sample_grids_k{K}`)，以便人工解释聚类内容。
    *   绘制 UMAP 散点图以可视化聚类结果 (`umap_kmeans_k{K}.png`)。
    *   绘制景点与聚类比例的热图 (`attraction_kmeans_k{K}_ratio_heatmap.png`)。
    *   生成每个聚类最集中的景点列表 (`kmeans_k{K}_cluster_top_attractions.csv`)。